# Stage 5 — Decision Thresholding, Post-Processing & Official Submission Packaging

This notebook implements the final stage of the Amazon ML Challenge 2026 entity resolution pipeline:

1. **Macro $F_{0.5}$ Threshold Optimization**: Fine-tuning decision threshold $\tau$ to balance the 2x precision penalty versus recall, ensuring singletons are correctly rewarded.
2. **Domain & Graph Consistency**: Enforcing zero cross-border matches and 1-to-1 source match capacity (preventing chain store branch confusion).
3. **Candidate Set Optimization (Scaling Bonus)**: Pruning low-probability candidates from `candidate_pairs.tsv` to achieve the smallest possible candidate set per Source 1 entity while preserving 100% of final matches.
4. **Format & Rule Validation**: Automated compliance checks using the competition's official `student_resource/utils/validate_submission.py`.
5. **Archive Packaging**: Compiling `output/submission.zip` with MD5/SHA256 checksums and manifest ready for portal upload.

In [ ]:
# 1. Environment & Path Setup
import os
import sys
import json
import time
import numpy as np
import pandas as pd

# Resolve project root dynamically
CWD = os.path.abspath(os.getcwd())
ROOT_DIR = CWD if os.path.exists(os.path.join(CWD, 'src')) else os.path.abspath(os.path.join(CWD, '..'))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from src.postprocessing import (
    filter_by_country_consistency,
    resolve_source_conflicts,
    build_final_matches_map,
    optimize_candidate_set,
    compute_candidate_efficiency_metrics,
    run_official_validator,
    package_submission_zip,
)
from src.matching.threshold import evaluate_f05, find_optimal_threshold

DATA_DIR = os.path.join(ROOT_DIR, 'student_resource', 'dataset')
PRE_DIR = os.path.join(ROOT_DIR, 'dataset', 'preprocessed')
VAL_GT = os.path.join(ROOT_DIR, 'dataset', 'val_split', 'val_ground_truth.tsv')
CAND_DIR = os.path.join(ROOT_DIR, 'dataset', 'candidates')
MODEL_DIR = os.path.join(ROOT_DIR, 'models')
OUT_DIR = os.path.join(ROOT_DIR, 'output')

os.makedirs(OUT_DIR, exist_ok=True)

print(f"Root Directory : {ROOT_DIR}")
print(f"Output Dir     : {OUT_DIR}")
print(f"Candidates Dir : {os.path.exists(CAND_DIR)}")

### 2. Threshold Sensitivity Analysis for Macro $F_{0.5}$
The official metric is Macro-averaged $F_{0.5}$:
$$F_{0.5} = \frac{1.25 \cdot \text{Precision} \cdot \text{Recall}}{0.25 \cdot \text{Precision} + \text{Recall}}$$
Because $\beta = 0.5$, precision errors are penalized **twice as heavily** as recall errors. A false merge ruins the entity cluster.

In [ ]:
# 2. Inspect Threshold Trade-off on Validation Split (if validation candidates exist)
val_cand_path = os.path.join(CAND_DIR, 'val_candidate_pairs.parquet')
if os.path.exists(val_cand_path) and os.path.exists(VAL_GT):
    val_cands = pd.read_parquet(val_cand_path)
    val_gt = pd.read_csv(VAL_GT, sep='\t', dtype=str, keep_default_na=False)
    gt_map = {r['source1_entity_id']: set(r['matched_entity_ids'].split(',')) - {''} for _, r in val_gt.iterrows()}
    
    # Simulated sweep over candidate_score (or model probabilities)
    score_col = 'candidate_score' if 'candidate_score' in val_cands.columns else val_cands.columns[-1]
    print(f"Sweeping decision thresholds over {len(val_cands):,} validation pairs...")
    sweep_results = []
    for tau in np.arange(0.50, 0.90, 0.05):
        passing = val_cands[val_cands[score_col] >= tau]
        preds = {}
        for qid, grp in passing.groupby('source1_entity_id'):
            preds[qid] = grp['candidate_entity_id'].tolist()
        res = evaluate_f05(preds, gt_map)
        sweep_results.append({'threshold': round(tau, 2), 'macro_f05': round(res['macro_f05'], 4), 'precision': round(res['micro_precision'], 4), 'recall': round(res['micro_recall'], 4)})
    display(pd.DataFrame(sweep_results))
else:
    print('Validation candidates not found. Run Stage 2 validation first.')

### 3. Stage 5 Graph & Domain Consistency Post-Processing
We enforce two competition domain rules:
1. **Country Integrity**: Filter out any candidate pairs that cross country boundaries (0 in 7.6M in ground truth).
2. **Capacity Constraint**: Limit each Source 1 query to at most **1 match in Source 2** and **1 match in Source 3** (resolves branch collisions in retail chains).

In [ ]:
# 3. Demonstrate Conflict Resolution on a Synthetic Query Pair
test_conflict_df = pd.DataFrame([
    {'source1_entity_id': 'S1_001', 'candidate_entity_id': 'S2_1001', 'probability': 0.88},
    {'source1_entity_id': 'S1_001', 'candidate_entity_id': 'S2_1002', 'probability': 0.74},
    {'source1_entity_id': 'S1_001', 'candidate_entity_id': 'S3_5001', 'probability': 0.92},
])
print('Raw candidate predictions (contains duplicate S2 candidates):')
display(test_conflict_df)

resolved = resolve_source_conflicts(test_conflict_df, threshold=0.70, max_per_source=1)
print('\nResolved predictions (retains only highest confidence candidate per source partition):')
display(resolved)

### 4. Candidate Budget Optimization (Scaling Rank Bonus)
> **Official Competition Notice:**
> *"The approach that generates a smaller candidate set per Source 1 entity will be ranked higher in the final evaluation beyond the public/private leaderboard."*

We optimize `candidate_pairs.tsv` by trimming the low-probability tail while **guaranteeing 100% preservation** of every match in `matching_results.tsv`.

In [ ]:
# 4. Generate & Validate Submission Files
from scripts.run_stage5 import main as run_stage5_pipeline

# Run Stage 5 CLI pipeline
matching_tsv = os.path.join(OUT_DIR, 'matching_results.tsv')
candidate_tsv = os.path.join(OUT_DIR, 'candidate_pairs.tsv')

if os.path.exists(matching_tsv) and os.path.exists(candidate_tsv):
    print('Submission files detected. Validating official rules...')
    valid_ok, valid_lines = run_official_validator(matching_tsv, candidate_path=candidate_tsv)
    for line in valid_lines:
        print(f"  {line}")
    
    # Package archive
    zip_path = os.path.join(OUT_DIR, 'submission.zip')
    manifest = package_submission_zip(matching_tsv, candidate_tsv, output_zip_path=zip_path)
    print(f"\nSubmission ready: {zip_path} ({manifest['zip_size_mb']} MB)")
else:
    print('Run `python scripts/run_stage5.py` to generate the test submission once Stage 2 test finishes.')